In [98]:
import pandas as pd
import os
import numpy as np

### Extraction

In [99]:
# This script lists all CSV files in the specified directory and stores their full paths in a list.
base_path = '../Data/Standrized_dataset/'
datasets_list = os.listdir(base_path)
full_datasets_path = []
for dataset in datasets_list:
    if dataset.endswith('.csv'):
        full_datasets_path.append(os.path.join(base_path, dataset))
    else:
        print(f"File {dataset} is not a CSV file and will be skipped.")
full_datasets_path

['../Data/Standrized_dataset/arc_dataset.csv',
 '../Data/Standrized_dataset/creative_problem_solving_dataset.csv',
 '../Data/Standrized_dataset/cs_dataset.csv',
 '../Data/Standrized_dataset/openbookqa_dataset.csv',
 '../Data/Standrized_dataset/quartz_dataset.csv',
 '../Data/Standrized_dataset/reclor_dataset.csv']

### Transfomation

### Mapping answers of MCQ datasets to indexes

In [100]:
def answers_mapping_to_index(answer) -> int:
    answer_types = [['A', 'B', 'D', 'C'], ['1', '2', '3', '4'], ['0.0', '1.0', '2.0', '3.0']]
    for i in range(len(answer_types)):
        for j in range(len(answer_types[i])):
            if answer == answer_types[i][j]:
                return j
    return 0

In [101]:
def dataset_transformation(dataset_path: str) -> pd.DataFrame:
    dataset = pd.read_csv(dataset_path)
    dataset_columns = dataset.columns
    if dataset['question_type'].iloc[0] == "open-ended":
        dataset.columns = ["id", "question", "answer", "question_type", "dimension"]
    elif dataset['question_type'].iloc[0] == "MCQ":
        dataset.columns = ["id", "question", "choices", "answer", "question_type", "dimension"]
    if "Unnamed: 0" in dataset_columns:
        dataset.rename(columns={"Unnamed: 0": "index"}, inplace=True)
    for column in dataset_columns:
        dataset.rename(columns={column: column.strip().lower()}, inplace=True)
    rows_duplicates = dataset.duplicated().sum()
    if rows_duplicates > 0:
        dataset.drop_duplicates(inplace=True)
    for column in dataset.columns:
        null_values = dataset[column].isnull().sum()
        if null_values > 0:
            dataset[column].dropna(inplace=True)
    if dataset['question_type'].unique() == "MCQ":
        dataset['answer_index'] = dataset['answer'].apply(answers_mapping_to_index).astype('int64')
    else:
        dataset['answer_index'] = 0
    return dataset

### Load

In [102]:
pd.read_csv('../Data/Standrized_dataset/creative_problem_solving_dataset.csv')

,Unnamed: 0,problem,solution,question_type,dimension
0,0,You spilled red wine on the hotel carpet and w...,Step1: Open the bottle of mineral water with t...,open-ended,Creativity
1,1,You accidentally locked your only pair of glas...,Step1: Use the remaining battery in your smart...,open-ended,Creativity
2,2,You have an important meeting but your suit is...,Step1: Hang the suit on the coat hanger on the...,open-ended,Creativity
3,3,The hotel bathroom door handle is broken and y...,Step 1: Unbend the wire hanger and flatten it ...,open-ended,Creativity
4,4,The hotel's WiFi signal is weak and you have a...,"Step1: Using the clothes hanger, create a hook...",open-ended,Creativity
...,...,...,...,...,...
1678,1678,You want to inflate a bike tire but the air pu...,Step1: Attach the balloon to the dust cleaner ...,open-ended,Creativity
1679,1679,You have a flat tire but do not have a tire ja...,Step1: Use the tire iron to loosen the lug nut...,open-ended,Creativity
1680,1680,You need to make fire in the porch for a barbe...,It is not possible to start a fire with the pr...,open-ended,Creativity
1681,1681,You want to inflate a bike tire but the air pu...,It is not possible to inflate the bike tire wi...,open-ended,Creativity


In [103]:
open_ended_datasets = {"question": [], "answer": [], "question_type": [], "dimension": [], "answer_index": []}
mcq_datasets = {"question": [], "choices": [], "answer": [], "question_type": [], "dimension": [], "answer_index": []}

for dataset_path in full_datasets_path:
    print(f"Processing dataset: {dataset_path}")
    dataset = dataset_transformation(dataset_path)
    if "question_type" in dataset.columns:
        if dataset["question_type"].iloc[0] == "open-ended":
            open_ended_datasets['question'].extend(dataset["question"].tolist())
            open_ended_datasets["answer"].extend(dataset["answer"].tolist())
            open_ended_datasets["question_type"].extend(dataset["question_type"].tolist())
            open_ended_datasets["dimension"].extend(dataset["dimension"].tolist())
            open_ended_datasets["answer_index"].extend(dataset["answer_index"].tolist())
        elif dataset["question_type"].iloc[0] == "MCQ":
            mcq_datasets['question'].extend(dataset["question"].tolist())
            mcq_datasets["choices"].extend(dataset["choices"].tolist())
            mcq_datasets["answer"].extend(dataset["answer"].tolist())
            mcq_datasets["question_type"].extend(dataset["question_type"].tolist())
            mcq_datasets["dimension"].extend(dataset["dimension"].tolist())
            mcq_datasets["answer_index"].extend(dataset["answer_index"].tolist())

open_ended_dataset = pd.DataFrame(data=open_ended_datasets)
mcq_dataset = pd.DataFrame(data=mcq_datasets)

Processing dataset: ../Data/Standrized_dataset/arc_dataset.csv
Processing dataset: ../Data/Standrized_dataset/creative_problem_solving_dataset.csv
Processing dataset: ../Data/Standrized_dataset/cs_dataset.csv
Processing dataset: ../Data/Standrized_dataset/openbookqa_dataset.csv
Processing dataset: ../Data/Standrized_dataset/quartz_dataset.csv
Processing dataset: ../Data/Standrized_dataset/reclor_dataset.csv


In [104]:
mcq_dataset = mcq_dataset.dropna()

In [105]:
print("Open-ended dataset:", open_ended_dataset.info())
print("\n")
print("MCQ dataset:", mcq_dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1855 entries, 0 to 1854
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1855 non-null   object
 1   answer         1855 non-null   object
 2   question_type  1855 non-null   object
 3   dimension      1855 non-null   object
 4   answer_index   1855 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 72.6+ KB
Open-ended dataset: None


<class 'pandas.core.frame.DataFrame'>
Index: 13410 entries, 0 to 13409
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       13410 non-null  object
 1   choices        13410 non-null  object
 2   answer         13410 non-null  object
 3   question_type  13410 non-null  object
 4   dimension      13410 non-null  object
 5   answer_index   13410 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 733.4+ KB
MCQ dataset: Non

In [106]:
final_dataset = pd.concat([open_ended_dataset, mcq_dataset], ignore_index=True)
final_dataset["choices"] = final_dataset["choices"].replace(np.nan, "No choices available")
print("null values in final dataset:\n", final_dataset.isnull().sum())

null values in final dataset:
 question         0
answer           0
question_type    0
dimension        0
answer_index     0
choices          0
dtype: int64


In [107]:
final_dataset.to_csv('../Data/final_dataset/final_dataset.csv', index=False)